In [1]:
from tarfile import PAX_NAME_FIELDS
import pandas as pd


import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
feature_df = pd.read_csv('./datasets/UCI_HAR_Dataset/dataset/features.txt', sep='\s+',header=None, names=['column_index','column_name'])

feature_name = feature_df.iloc[:,1].values.tolist()
print('전체 피저명엥서 10개만 추출 :',feature_name[:10])

전체 피저명엥서 10개만 추출 : ['tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z', 'tBodyAcc-std()-X', 'tBodyAcc-std()-Y', 'tBodyAcc-std()-Z', 'tBodyAcc-mad()-X', 'tBodyAcc-mad()-Y', 'tBodyAcc-mad()-Z', 'tBodyAcc-max()-X']


<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\claire_lee\AppData\Local\Temp\ipykernel_19060\2671479924.py:1: SyntaxWarning: invalid escape sequence '\s'
  feature_df = pd.read_csv('./datasets/UCI_HAR_Dataset/dataset/features.txt', sep='\s+',header=None, names=['column_index','column_name'])


In [3]:
f_dup_df = feature_df.groupby('column_name').count()
print(f_dup_df[f_dup_df['column_index']>1].count())
f_dup_df[f_dup_df['column_index']>1].head()

column_index    42
dtype: int64


,column_index
column_name,
"fBodyAcc-bandsEnergy()-1,16",3
"fBodyAcc-bandsEnergy()-1,24",3
"fBodyAcc-bandsEnergy()-1,8",3
"fBodyAcc-bandsEnergy()-17,24",3
"fBodyAcc-bandsEnergy()-17,32",3


In [4]:
def get_new_feature_name_df(old_feature_name_df):
    f_dup_df = pd.DataFrame(data=old_feature_name_df.groupby('column_name').cumcount(), columns=['dup_cnt'])
    f_dup_df = f_dup_df.reset_index()
    new_feature_name_df = pd.merge(old_feature_name_df.reset_index(), f_dup_df, how='outer')
    new_feature_name_df['column_name'] = new_feature_name_df[['column_name', 'dup_cnt']].apply(lambda x : x[0]+'_'+str(x[1]) if x[1] > 0 else x[0], axis=1)
    new_feature_name_df = new_feature_name_df.drop(['index'], axis=1)
    return new_feature_name_df

In [5]:
def get_humandataset():
    feature_df = pd.read_csv('./datasets/UCI_HAR_Dataset/dataset/features.txt', sep='\s+', header=None, names=['column_index','column_name'])
    new_feature_name_df = get_new_feature_name_df(feature_df)

    feature_name = new_feature_name_df.iloc[:, 1].values.tolist()
    
    x_train = pd.read_csv('./datasets/UCI_HAR_Dataset/dataset/train/X_train.txt', sep='\s+', names=feature_name)
    x_test = pd.read_csv('./datasets/UCI_HAR_Dataset/dataset/test/X_test.txt', sep='\s+', names=feature_name)
    
    y_train = pd.read_csv('./datasets/UCI_HAR_Dataset/dataset/train/y_train.txt', sep='\s+',header=None, names=['action'])#
    y_test = pd.read_csv('./datasets/UCI_HAR_Dataset/dataset/test/y_test.txt', sep='\s+', header=None, names=['action'])

    return x_train, x_test, y_train, y_test

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\s'
<>:8: SyntaxWarning: invalid escape sequence '\s'
<>:10: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\s'
<>:8: SyntaxWarning: invalid escape sequence '\s'
<>:10: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\s'
C:\Users\claire_lee\AppData\Local\Temp\ipykernel_19060\348826552.py:2: SyntaxWarning: invalid escape sequence '\s'
  feature_df = pd.read_csv('./datasets/UCI_HAR_Dataset/dataset/features.txt', sep='\s+', header=None, names=['column_index','column_name'])
C:\Users\claire_lee\AppData\Local\Temp\ipykernel_19060\348826552.py:7: SyntaxWarning: invalid escape sequence '\s'
  x_train = pd.read_csv('./datasets/UCI_HAR_Dataset/dataset/train/X_train.txt', sep='\s+', names=feature_name)
C:\Users\claire

In [6]:
x_train, x_test, y_train, y_test = get_humandataset()

C:\Users\claire_lee\AppData\Local\Temp\ipykernel_19060\788222927.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  new_feature_name_df['column_name'] = new_feature_name_df[['column_name', 'dup_cnt']].apply(lambda x : x[0]+'_'+str(x[1]) if x[1] > 0 else x[0], axis=1)


In [7]:
print('## 학습 피처 데이터셋 info()')

#x_train = x_train.drop(columns=['action'], errors='ignore')
#x_test = x_test.drop(columns=['action'], errors='ignore')
print(x_train.info())
print(x_test.info())

## 학습 피처 데이터셋 info()
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7352 entries, 0 to 7351
Columns: 561 entries, tBodyAcc-mean()-X to angle(Z,gravityMean)
dtypes: float64(561)
memory usage: 31.5 MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2947 entries, 0 to 2946
Columns: 561 entries, tBodyAcc-mean()-X to angle(Z,gravityMean)
dtypes: float64(561)
memory usage: 12.6 MB
None


In [8]:
#y_train.head()
print(y_train['action'].value_counts())

action
6    1407
5    1374
4    1286
1    1226
2    1073
3     986
Name: count, dtype: int64


In [9]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import numpy as np
from scipy import sparse

In [10]:
if sparse.issparse(x_train):
    x_train = x_train.toarray()

if sparse.issparse(x_test):
    x_test = x_test.toarray()

In [11]:
dt_clf = DecisionTreeClassifier(random_state=156)
dt_clf.fit(x_train, y_train['action'])
if 'action' in x_test.columns:
     x_test = x_test.drop('action', axis=1)
     x_test = x_test[x_train.columns]

#print(x_test['action'].info)
#x_test2 = np.array(x_test)
pred = dt_clf.predict(x_test)
accuracy = accuracy_score(y_test, pred)
print('결정 트리 예측 정확도 : {0:.4f}'.format(accuracy))

print('DecisionTreeClassifier 기본 하이퍼 파라미터 : \n',dt_clf.get_params())

결정 트리 예측 정확도 : 0.8548
DecisionTreeClassifier 기본 하이퍼 파라미터 : 
 {'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'random_state': 156, 'splitter': 'best'}


In [12]:
print(y_train['action'].value_counts())

action
6    1407
5    1374
4    1286
1    1226
2    1073
3     986
Name: count, dtype: int64


In [27]:
from sklearn.model_selection import GridSearchCV

params = {
    'max_depth' : [6,8,10,12,16,20,24],
    'min_samples_split':[16]
}

grid_cv = GridSearchCV(dt_clf, param_grid=params, scoring='accuracy', cv=5, verbose=1)
grid_cv.fit(x_train, y_train)

Fitting 5 folds for each of 7 candidates, totalling 35 fits


GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=156),
             param_grid={'max_depth': [6, 8, 10, 12, 16, 20, 24],
                         'min_samples_split': [16]},
             scoring='accuracy', verbose=1)

In [32]:
cv_results_df=pd.DataFrame(grid_cv.cv_results_)

cv_results_df[['param_max_depth','mean_test_score']]

,param_max_depth,mean_test_score
0,6,0.847662
1,8,0.854879
2,10,0.852705
3,12,0.845768
4,16,0.847127
5,20,0.848624
6,24,0.848624
